# Smart Trip Planning and Recommendation System - EDA & Preprocessing
### College Data Science Project
This notebook covers:
1. **Loading CSV Datasets** via Pandas (`destinations`, `attractions`, `hotels`, `restaurants`, `transport`)
2. **Data Cleaning**: Missing value handling, duplicate checks, data type casting
3. **Feature Engineering**: Text tokenization, composite tags, word counts, and min-max normalization
4. **Exploratory Data Analysis (EDA) Visualizations**:
   - Attraction Rating Distribution
   - Entry Ticket Price Distribution
   - Destination Popularity Ranking
   - Attraction Categories Distribution
   - Average Rating by Destination


In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Styling configurations
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
DATA_DIR = '../data'
print('Libraries successfully imported.')


## 1. Load Datasets Using Pandas


In [ ]:
destinations = pd.read_csv(os.path.join(DATA_DIR, 'destinations.csv'))
attractions = pd.read_csv(os.path.join(DATA_DIR, 'attractions.csv'))
hotels = pd.read_csv(os.path.join(DATA_DIR, 'hotels.csv'))
restaurants = pd.read_csv(os.path.join(DATA_DIR, 'restaurants.csv'))
transport = pd.read_csv(os.path.join(DATA_DIR, 'transport.csv'))

print(f'Destinations shape: {destinations.shape}')
print(f'Attractions shape:  {attractions.shape}')
print(f'Hotels shape:       {hotels.shape}')
print(f'Restaurants shape:  {restaurants.shape}')
print(f'Transport shape:    {transport.shape}')
attractions.head(3)


## 2. Data Cleaning: Missing Values, Duplicates & Type Correction


In [ ]:
# Missing Values Check
print('--- Missing Values in Attractions ---')
print(attractions.isnull().sum())

# Fill any potential missing values
attractions['description'] = attractions['description'].fillna('Popular tourist attraction')
attractions['tags'] = attractions['tags'].fillna('travel, sightseeing')
attractions['price'] = attractions['price'].fillna(0.0)
attractions['rating'] = attractions['rating'].fillna(attractions['rating'].median())

# Duplicate Rows Removal
dups = attractions.duplicated().sum()
print(f'Duplicate rows detected: {dups}')
if dups > 0:
    attractions.drop_duplicates(inplace=True)

# Data Type Corrections
attractions['id'] = attractions['id'].astype(int)
attractions['rating'] = attractions['rating'].astype(float)
attractions['price'] = attractions['price'].astype(float)
attractions['duration'] = attractions['duration'].astype(float)
attractions['popularity'] = attractions['popularity'].astype(int)
attractions['latitude'] = attractions['latitude'].astype(float)
attractions['longitude'] = attractions['longitude'].astype(float)
attractions.dtypes


## 3. Feature Engineering & Normalization


In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower())
    return re.sub(r'\s+', ' ', text).strip()

# Combine category, tags, and description with weighted repetition
attractions['combined_features'] = attractions.apply(
    lambda r: clean_text(f"{r['category']} {r['category']} {r['tags']} {r['tags']} {r['description']} {r['indoor_outdoor']}"),
    axis=1
)

attractions['desc_word_count'] = attractions['description'].apply(lambda x: len(str(x).split()))
attractions['is_free'] = (attractions['price'] == 0).astype(int)

# Min-Max Normalization of Rating and Popularity to [0, 1]
attractions['norm_rating'] = ((attractions['rating'] - 3.5) / (5.0 - 3.5)).clip(0, 1)
attractions['norm_popularity'] = ((attractions['popularity'] - 50) / (100 - 50)).clip(0, 1)

attractions[['name', 'category', 'desc_word_count', 'norm_rating', 'norm_popularity', 'combined_features']].head(3)


## 4. EDA Visualization 1: Rating Distribution


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(attractions['rating'], kde=True, color='#2563eb', bins=8, edgecolor='black', alpha=0.75)
plt.title(f'Attraction Rating Distribution (Mean: {attractions["rating"].mean():.2f} Stars)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Rating (1.0 to 5.0)', fontsize=11)
plt.ylabel('Count of Attractions', fontsize=11)
plt.axvline(attractions['rating'].mean(), color='#dc2626', linestyle='--', linewidth=2, label=f'Mean: {attractions["rating"].mean():.2f}')
plt.legend()
plt.tight_layout()
plt.show()


## 5. EDA Visualization 2: Entry Ticket Price Distribution


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(attractions['price'], kde=False, color='#10b981', bins=12, edgecolor='black', alpha=0.75)
plt.title(f'Attraction Ticket Price Distribution in INR (Free: {attractions["is_free"].sum()} Attractions)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Ticket Price (INR)', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.tight_layout()
plt.show()


## 6. EDA Visualization 3: Destination Popularity Ranking


In [ ]:
plt.figure(figsize=(10, 5))
dest_pop = attractions.groupby('destination')['popularity'].mean().sort_values(ascending=False)
palette = sns.color_palette('Blues_r', len(dest_pop))
bars = plt.bar(dest_pop.index, dest_pop.values, color=palette, edgecolor='black', alpha=0.85)
plt.title('Destination Popularity Ranking (Average Attraction Popularity)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Destination', fontsize=11)
plt.ylabel('Mean Popularity Score (0-100)', fontsize=11)
plt.ylim(70, 100)
for bar in bars:
    y = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, y + 0.6, f'{y:.1f}', ha='center', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.show()


## 7. EDA Visualization 4: Attractions by Category Breakdown


In [ ]:
plt.figure(figsize=(11, 5))
cat_counts = attractions['category'].value_counts()
sns.barplot(x=cat_counts.index, y=cat_counts.values, hue=cat_counts.index, palette='viridis', edgecolor='black', alpha=0.85, legend=False)
plt.title(f'Attractions by Category Breakdown (Total: {len(attractions)})', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Category', fontsize=11)
plt.ylabel('Number of Attractions', fontsize=11)
plt.xticks(rotation=35, ha='right')
for i, v in enumerate(cat_counts.values):
    plt.text(i, v + 0.2, str(v), ha='center', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.show()


## 8. EDA Visualization 5: Average Rating by Destination


In [ ]:
plt.figure(figsize=(10, 5))
avg_rating = attractions.groupby('destination')['rating'].mean().sort_values(ascending=False)
sns.barplot(x=avg_rating.index, y=avg_rating.values, hue=avg_rating.index, palette='magma', edgecolor='black', alpha=0.85, legend=False)
plt.title('Average Attraction Rating by Destination', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Destination', fontsize=11)
plt.ylabel('Average Rating (Stars)', fontsize=11)
plt.ylim(4.0, 5.0)
for i, v in enumerate(avg_rating.values):
    plt.text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.show()
